### Steps:
1. Importing the Libraries
2. Importing the Dataset
3. Data Analysis -> DE,DM,DC,DV,EDA
4. Feature Engineering -> Encoding, Scaling
5. Splitting the data using CV
6. Model Selection - KNN
7. [Optional] HyperParameter Tuning
8. Training the model
9. Testing the model
10. Performance Analysis - ConfusionMatrix

---------

1. Importing the libraries

In [759]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.preprocessing import StandardScaler

2. Importing the Datasets

- Credits : https://drive.google.com/file/d/1nWdt8_Hh-M35N4GwRixPmb3AsHNQVZKQ/view?usp=sharing
- Movies : https://drive.google.com/file/d/1HxZXAkPajH40pChWvpAseOi71VW0k3wS/view?usp=sharing

In [760]:
movies=pd.read_csv('imdb_5000_movies.csv')
credits=pd.read_csv('imdb_5000_credits.csv')

In [761]:
movies.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [762]:
credits.head(3)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [763]:
movies.describe()

,budget,id,popularity,revenue,runtime,vote_average,vote_count
count,4.803000e+03,4803.000000,4803.000000,4.803000e+03,4801.000000,4803.000000,4803.000000
mean,2.904504e+07,57165.484281,21.492301,8.226064e+07,106.875859,6.092172,690.217989
std,4.072239e+07,88694.614033,31.816650,1.628571e+08,22.611935,1.194612,1234.585891
min,0.000000e+00,5.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,7.900000e+05,9014.500000,4.668070,0.000000e+00,94.000000,5.600000,54.000000
50%,1.500000e+07,14629.000000,12.921594,1.917000e+07,103.000000,6.200000,235.000000
75%,4.000000e+07,58610.500000,28.313505,9.291719e+07,118.000000,6.800000,737.000000
max,3.800000e+08,459488.000000,875.581305,2.787965e+09,338.000000,10.000000,13752.000000


In [764]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [765]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [766]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [767]:
movies.genres[0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [768]:
l=[]
m=[]
i=movies.genres[0]
i=re.sub('"name": "','',i)
i=re.sub('"id": ','',i)
i=re.sub('"','',i)
i=re.sub('\[','',i)
i=re.sub('\]','',i)
i=re.sub('\{','',i)
i=re.sub('\}','',i)
for k in i.split(' '):
    k=re.sub(',','',k)
    l.append(k)
for j in range(1,len(l),2):
    m.append(l[j])
m

['Action', 'Adventure', 'Fantasy', 'Science']

In [769]:
re.search('"name": "',i)

Using re

In [770]:
for p in range(len(movies.genres)):
    l=[]
    m=[]
    i=movies.genres[p]
    i=re.sub('"name": "','',i)
    i=re.sub('"id": ','',i)
    i=re.sub('"','',i)
    i=re.sub('\[','',i)
    i=re.sub('\]','',i)
    i=re.sub('\{','',i)
    i=re.sub('\}','',i)
    for k in i.split(' '):
        k=re.sub(',','',k)
        l.append(k)
    for j in range(1,len(l),2):
        m.append(l[j])
    print(m)

['Action', 'Adventure', 'Fantasy', 'Science']
['Adventure', 'Fantasy', 'Action']
['Action', 'Adventure', 'Crime']
['Action', 'Crime', 'Drama', 'Thriller']
['Action', 'Adventure', 'Science']
['Fantasy', 'Action', 'Adventure']
['Animation', 'Family']
['Action', 'Adventure', 'Science']
['Adventure', 'Fantasy', 'Family']
['Action', 'Adventure', 'Fantasy']
['Adventure', 'Fantasy', 'Action', 'Science']
['Adventure', 'Action', 'Thriller', 'Crime']
['Adventure', 'Fantasy', 'Action']
['Action', 'Adventure', 'Western']
['Action', 'Adventure', 'Fantasy', 'Science']
['Adventure', 'Family', 'Fantasy']
['Science', '28', '12']
['Adventure', 'Action', 'Fantasy']
['Action', 'Comedy', 'Science']
['Action', 'Adventure', 'Fantasy']
['Action', 'Adventure', 'Fantasy']
['Action', 'Adventure']
['Adventure', 'Fantasy']
['Adventure', 'Fantasy']
['Adventure', 'Drama', 'Action']
['Drama', 'Romance', 'Thriller']
['Adventure', 'Action', 'Science']
['Thriller', 'Action', 'Adventure', 'Science']
['Action', 'Adventure

Using json

In [771]:
genres=[]
for i in range(len(movies.genres)):
    genre=[]
    a=eval(movies.genres[i])
    for j in a:
        genre.append(j['name'])
    genres.append(genre)

In [772]:
movies.genres=pd.Series(genres)

In [773]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[Adventure, Fantasy, Action]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


Converting spoken_languages,production countries,production companies and keywords

In [774]:
keyword=[]
for i in range(len(movies.keywords)):
    k=[]
    a=eval(movies.keywords[i])
    for j in a:
        k.append(j['name'])
    keyword.append(k)

In [775]:
prod_comp=[]
for i in range(len(movies.production_companies)):
    k=[]
    a=eval(movies.production_companies[i])
    for j in a:
        k.append(j['name'])
    prod_comp.append(k)

In [776]:
prod_cont=[]
for i in range(len(movies.production_countries)):
    k=[]
    a=eval(movies.production_countries[i])
    for j in a:
        k.append(j['name'])
    prod_cont.append(k)

In [777]:
lang=[]
for i in range(len(movies.spoken_languages)):
    k=[]
    a=eval(movies.spoken_languages[i])
    for j in a:
        lang.append(j['name'])

In [778]:
movies.keywords=pd.Series(keyword)
movies.production_companies=pd.Series(prod_comp)
movies.production_countries=pd.Series(prod_cont)
movies.spoken_languages=pd.Series(lang)

In [779]:
movies['year']=pd.DatetimeIndex(movies.release_date).year
movies['month']=pd.DatetimeIndex(movies.release_date).month
movies['day']=pd.DatetimeIndex(movies.release_date).day

In [780]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,year,month,day
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",http://www.avatarmovie.com/,19995,"[culture clash, future, space war, space colon...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[Ingenious Film Partners, Twentieth Century Fo...",...,162.0,English,Released,Enter the World of Pandora.,Avatar,7.2,11800,2009.0,12.0,10.0
1,300000000,"[Adventure, Fantasy, Action]",http://disney.go.com/disneypictures/pirates/,285,"[ocean, drug abuse, exotic island, east india ...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[Walt Disney Pictures, Jerry Bruckheimer Films...",...,169.0,Español,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,2007.0,5.0,19.0


Random value

In [781]:
movies.iloc[140]

budget                                                          150000000
genres                                          [Action, Drama, Thriller]
homepage                                                              NaN
id                                                                 117251
keywords                [usa president, conspiracy, secret service, th...
original_language                                                      en
original_title                                           White House Down
overview                Capitol Policeman John Cale has just been deni...
popularity                                                      39.004588
production_companies    [Columbia Pictures, Centropolis Entertainment,...
production_countries                           [United States of America]
release_date                                                   2013-06-27
revenue                                                         205366737
runtime                               

In [782]:
movies.isnull().sum()

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
year                       1
month                      1
day                        1
dtype: int64

Data Manipulation

In [783]:
movies.tagline.nunique()

3944

In [784]:
movies.drop(columns=['homepage','tagline','original_title','overview','title','release_date'],inplace=True)

In [785]:
movies.head(1)

,budget,genres,id,keywords,original_language,popularity,production_companies,production_countries,revenue,runtime,spoken_languages,status,vote_average,vote_count,year,month,day
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",en,150.437577,"[Ingenious Film Partners, Twentieth Century Fo...","[United States of America, United Kingdom]",2787965087,162.0,English,Released,7.2,11800,2009.0,12.0,10.0


In [786]:
movies.isnull().sum()

budget                  0
genres                  0
id                      0
keywords                0
original_language       0
popularity              0
production_companies    0
production_countries    0
revenue                 0
runtime                 2
spoken_languages        0
status                  0
vote_average            0
vote_count              0
year                    1
month                   1
day                     1
dtype: int64

In [787]:
movies.dropna(inplace=True)

In [788]:
movies.shape

(4800, 17)

In [789]:
movies.isnull().sum()

budget                  0
genres                  0
id                      0
keywords                0
original_language       0
popularity              0
production_companies    0
production_countries    0
revenue                 0
runtime                 0
spoken_languages        0
status                  0
vote_average            0
vote_count              0
year                    0
month                   0
day                     0
dtype: int64

In [790]:
movies.head(1)

,budget,genres,id,keywords,original_language,popularity,production_companies,production_countries,revenue,runtime,spoken_languages,status,vote_average,vote_count,year,month,day
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",en,150.437577,"[Ingenious Film Partners, Twentieth Century Fo...","[United States of America, United Kingdom]",2787965087,162.0,English,Released,7.2,11800,2009.0,12.0,10.0


Credits Dataset

In [791]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [792]:
credits.isnull().sum()

movie_id    0
title       0
cast        0
crew        0
dtype: int64

In [793]:
movies=pd.merge(movies,credits,left_on='id',right_on='movie_id')

In [794]:
movies.head(3)

,budget,genres,id,keywords,original_language,popularity,production_companies,production_countries,revenue,runtime,...,status,vote_average,vote_count,year,month,day,movie_id,title,cast,crew
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",en,150.437577,"[Ingenious Film Partners, Twentieth Century Fo...","[United States of America, United Kingdom]",2787965087,162.0,...,Released,7.2,11800,2009.0,12.0,10.0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[Adventure, Fantasy, Action]",285,"[ocean, drug abuse, exotic island, east india ...",en,139.082615,"[Walt Disney Pictures, Jerry Bruckheimer Films...",[United States of America],961000000,169.0,...,Released,6.9,4500,2007.0,5.0,19.0,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[Action, Adventure, Crime]",206647,"[spy, based on novel, secret agent, sequel, mi...",en,107.376788,"[Columbia Pictures, Danjaq, B24]","[United Kingdom, United States of America]",880674609,148.0,...,Released,6.3,4466,2015.0,10.0,26.0,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [795]:
movies.isnull().sum().sum()

0

- Cast Name
- Crew Name
- Crew Department
- Crew Director

In [796]:
movies.cast[0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

Cast Name

In [797]:
cast=[]
for i in range(len(movies.cast)):
    k=[]
    a=eval(movies.cast[i])
    for j in a:
        k.append(j['name'])
    cast.append(k)

Crew Name

In [798]:
crew=[]
for i in range(len(movies.crew)):
    k=[]
    a=eval(movies.crew[i])
    for j in a:
        k.append(j['name'])
    crew.append(k)

Crew Department

In [801]:
dept=[]
for i in range(len(movies.crew)):
    k=[]
    a=eval(movies.crew[i])
    for j in a:
        k.append(j['department'])
    dept.append(k)

Crew Director

In [829]:
director=[]
for i in range(len(movies.crew)):
    a=eval(movies.crew[i])
    k=[]
    for j in a:
        if j['job']=='Director':
            k.append(j['name'])
    if len(k)>=1:
        director.append(k[0])
    else:
        director.append(None)
    k=[]

In [835]:
movies['Cast']=(cast)
movies['Crew']=(crew)
movies['Department']=dept
movies['Director']=director

In [838]:
movies.drop(columns=['cast','crew'],inplace=True)

In [839]:
movies.head(1)

,budget,genres,id,keywords,original_language,popularity,production_companies,production_countries,revenue,runtime,...,vote_count,year,month,day,movie_id,title,Cast,Crew,Department,Director
0,237000000,"[Action, Adventure, Fantasy, Science Fiction]",19995,"[culture clash, future, space war, space colon...",en,150.437577,"[Ingenious Film Partners, Twentieth Century Fo...","[United States of America, United Kingdom]",2787965087,162.0,...,11800,2009.0,12.0,10.0,19995,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Stephen E. Rivkin, Rick Carter, Christopher B...","[Editing, Art, Sound, Sound, Production, Sound...",James Cameron


Making a cleaned copy

In [844]:
Cleaned_movies=movies.copy()

Required Columns : [Budget, genres, keywords, popularity, revenue,runtime,title,vote_count, year,month,day,director,cast]